# Mesh Setup Comparison: Your Implementation vs. Official DeepMind Notebook

This notebook compares your current JAX mesh setup for multi-GPU training with the official DeepMind AlphaGenome finetuning notebook.

**Goal:** Identify differences causing NCCL errors in your implementation.

**Status:** `jax.eval_shape` validation **PASSED ✓** - shapes are compatible, issue is in mesh/sharding configuration.

## 1. Official DeepMind Mesh Setup

From the official AlphaGenome finetuning notebook (`scripts/ag-colab_ft.py`):

In [1]:
# Official DeepMind Implementation
# From: scripts/ag-colab_ft.py lines 210-265

official_code = '''
# Set Up Device Mesh for Data Parallelism
num_devices = jax.local_device_count()
devices = mesh_utils.create_device_mesh((num_devices,))
mesh = Mesh(devices, axis_names=(\'data\',))
data_sharding = P(\'data\')
replicated_sharding = P()

# Initialize New Output Heads
forward_fn = finetune.get_forward_fn(
    output_metadata, jmp_policy=\'params=float32,compute=float32,output=float32\'
)
with jax.set_mesh(mesh):
  batch = jax.device_put(batch, data_sharding)
  params_ft, state_ft = jax.jit(
      forward_fn.init,
      in_shardings=(replicated_sharding, data_sharding),
      out_shardings=replicated_sharding,
  )(jax.random.PRNGKey(0), batch)

# Setup optimizer and train_step
optimizer = optax.chain(
    optax.clip_by_global_norm(0.5),
    optax.adam(LEARNING_RATE),
)
opt_state = optimizer.init(params)
train_step = jax.jit(
    finetune.get_train_step(forward_fn.apply, optimizer),
    in_shardings=(
        replicated_sharding,
        replicated_sharding,
        replicated_sharding,
        data_sharding,
    ),
    out_shardings=(
        replicated_sharding,
        replicated_sharding,
        replicated_sharding,
        replicated_sharding,
    ),
)

# Training loop
with jax.set_mesh(mesh):
    batch = jax.device_put(batch, data_sharding)
    params, state, opt_state, scalars = train_step(
        params, state, opt_state, batch
    )
'''

print("Official DeepMind Mesh Setup:")
print("="*60)
print(official_code)

Official DeepMind Mesh Setup:

# Set Up Device Mesh for Data Parallelism
num_devices = jax.local_device_count()
devices = mesh_utils.create_device_mesh((num_devices,))
mesh = Mesh(devices, axis_names=('data',))
data_sharding = P('data')
replicated_sharding = P()

# Initialize New Output Heads
forward_fn = finetune.get_forward_fn(
    output_metadata, jmp_policy='params=float32,compute=float32,output=float32'
)
with jax.set_mesh(mesh):
  batch = jax.device_put(batch, data_sharding)
  params_ft, state_ft = jax.jit(
      forward_fn.init,
      in_shardings=(replicated_sharding, data_sharding),
      out_shardings=replicated_sharding,
  )(jax.random.PRNGKey(0), batch)

# Setup optimizer and train_step
optimizer = optax.chain(
    optax.clip_by_global_norm(0.5),
    optax.adam(LEARNING_RATE),
)
opt_state = optimizer.init(params)
train_step = jax.jit(
    finetune.get_train_step(forward_fn.apply, optimizer),
    in_shardings=(
        replicated_sharding,
        replicated_sharding,
      

### Key Features of Official Setup:

1. **Uses `mesh_utils.create_device_mesh((num_devices,))`** instead of `jax.devices()`
2. **Axis name is `'data'`** not `'batch'`
3. **Uses `jax.set_mesh(mesh)` context manager** around device_put and training
4. **Sharding is simple:** `P('data')` for data, `P()` for replicated
5. **Explicit `in_shardings` and `out_shardings` in all `jax.jit()` calls**

## 2. Your Current Mesh Setup

From your current implementation in `models/model.py` and `scripts/pytorch_ft.py`:

In [ ]:
# Your Current Implementation
# From: scripts/pytorch_ft.py lines 250-260 and models/model.py lines 505-540

your_code = '''
# Create device mesh for data parallelism
mesh = Mesh(jax.devices(), (\'batch\',))

# Shard batch dimension across devices
batch_sharding = NamedSharding(mesh, P(\'batch\', None, None))  # [batch, seq_len, 4]
organism_sharding = NamedSharding(mesh, P(\'batch\'))           # [batch]
signal_sharding = NamedSharding(mesh, P(\'batch\', None))       # [batch, seq_len]
param_sharding = NamedSharding(mesh, P())                      # replicated

# Replicate parameters across all devices
self.params = jax.device_put(self.params, self.param_sharding)
self.state = jax.device_put(self.state, self.param_sharding)
self.opt_state = jax.device_put(self.opt_state, self.param_sharding)

# JIT-compiled train step with explicit sharding specifications
self._train_step_jit = jax.jit(
    self._train_step,
    static_argnames=[\'loss_fn\'],
    in_shardings=(
        batch_sharding,                      # dna_sequence
        organism_sharding,                   # organism_index
        {\'signal\': signal_sharding},        # targets dict
        replicated,                          # params
        replicated,                          # state
        replicated,                          # opt_state
    ),
    out_shardings=(
        replicated,                          # loss_value
        None,                                # predictions (let JAX infer)
        replicated,                          # new_params
        replicated,                          # new_opt_state
        replicated,                          # grad_norm
    )
)

# Training loop (NO jax.set_mesh context manager)
for batch in data_loader:
    loss, predictions, grad_norm = model.train_step(
        batch[\'dna_sequence\'],
        batch[\'organism_index\'],
        batch[\'targets\'],
        loss_fn
    )
'''

print("Your Current Mesh Setup:")
print("="*60)
print(your_code)

## 3. Side-by-Side Comparison Table

Critical differences between implementations:

In [2]:
import pandas as pd

comparison_data = {
    'Feature': [
        'Device mesh creation',
        'Axis name',
        'Data sharding pattern',
        'Parameter sharding',
        'jax.set_mesh() usage',
        'Batch device_put location',
        'in_shardings specification',
        'out_shardings specification',
        'Sharding complexity',
    ],
    'Official DeepMind': [
        'mesh_utils.create_device_mesh((n,))',
        "'data'",
        "P('data')",
        'P()',
        'YES - wraps device_put and train_step',
        'Inside jax.set_mesh(mesh) context',
        '(repl, repl, repl, data)',
        '(repl, repl, repl, repl)',
        'Simple: P(\'data\') or P()',
    ],
    'Your Implementation': [
        'Mesh(jax.devices(), ...)',
        "'batch'",
        "P('batch', None, None)",
        'P()',
        'NO - no context manager',
        'Outside any context manager',
        '(batch_3d, batch_1d, {dict}, repl, repl, repl)',
        '(repl, None, repl, repl, repl)',
        "Complex: P('batch', None, None)",
    ],
    'Impact': [
        '⚠️ Different device ordering',
        '✓ Name only (cosmetic)',
        '⚠️⚠️ CRITICAL - dimension mismatch',
        '✓ Same (replicated)',
        '⚠️⚠️ CRITICAL - missing context',
        '⚠️⚠️ CRITICAL - outside mesh context',
        '⚠️ More complex, dict input',
        '⚠️ None for predictions (inferred)',
        '⚠️ May confuse JAX sharding inference',
    ]
}

df = pd.DataFrame(comparison_data)
print(df.to_string(index=False))

                    Feature                     Official DeepMind                            Your Implementation                                Impact
       Device mesh creation   mesh_utils.create_device_mesh((n,))                       Mesh(jax.devices(), ...)          ⚠️ Different device ordering
                  Axis name                                'data'                                        'batch'                ✓ Name only (cosmetic)
      Data sharding pattern                             P('data')                         P('batch', None, None)    ⚠️⚠️ CRITICAL - dimension mismatch
         Parameter sharding                                   P()                                            P()                   ✓ Same (replicated)
       jax.set_mesh() usage YES - wraps device_put and train_step                        NO - no context manager       ⚠️⚠️ CRITICAL - missing context
  Batch device_put location     Inside jax.set_mesh(mesh) context                    Outside a

## 4. Critical Issues Identified

### ⚠️⚠️ Issue 1: Missing `jax.set_mesh()` Context Manager

**Official:** Wraps ALL device_put and training calls in `with jax.set_mesh(mesh):`  
**Yours:** No context manager - mesh is created but not set as active  
**Impact:** JAX cannot track mesh context, may insert implicit resharding operations  

**Fix:**
```python
with jax.set_mesh(mesh):
    for batch in data_loader:
        batch = jax.device_put(batch, data_sharding)
        loss, predictions, grad_norm = model.train_step(...)
```

### ⚠️⚠️ Issue 2: Complex Sharding Specifications

**Official:** Simple `P('data')` for batch dimension only  
**Yours:** Explicit `P('batch', None, None)` for all dimensions  
**Impact:** May confuse JAX's sharding propagation logic  

**Fix:**
```python
# Simplify from:
batch_sharding = NamedSharding(mesh, P('batch', None, None))

# To:
data_sharding = NamedSharding(mesh, P('batch'))
# Let JAX infer replication for other dimensions
```

### ⚠️ Issue 3: Not Using `mesh_utils.create_device_mesh()`

**Official:** Uses `mesh_utils.create_device_mesh((num_devices,))`  
**Yours:** Uses `Mesh(jax.devices(), ...)`  
**Impact:** `create_device_mesh()` may set up better device topology  

**Fix:**
```python
from jax.experimental import mesh_utils

num_devices = jax.local_device_count()
devices = mesh_utils.create_device_mesh((num_devices,))
mesh = Mesh(devices, axis_names=('batch',))
```

## 5. Recommended Code Changes

### Update `scripts/pytorch_ft.py` or training script:

In [ ]:
print("BEFORE (Current Implementation):")
print("="*60)
print('''
# Create device mesh
mesh = Mesh(jax.devices(), ('batch',))
batch_sharding = NamedSharding(mesh, P('batch', None, None))
organism_sharding = NamedSharding(mesh, P('batch'))
signal_sharding = NamedSharding(mesh, P('batch', None))

# Training loop
for batch in data_loader:
    loss, predictions, grad_norm = model.train_step(
        batch['dna_sequence'],
        batch['organism_index'],
        batch['targets'],
        loss_fn
    )
''')

print("\n" + "="*60)
print(" AFTER (Fixed Implementation):")
print("="*60)
print('''
# Import mesh_utils
from jax.experimental import mesh_utils

# Create device mesh (official way)
num_devices = jax.local_device_count()
devices = mesh_utils.create_device_mesh((num_devices,))
mesh = Mesh(devices, axis_names=('data',))

# Simplified sharding specs
data_sharding = NamedSharding(mesh, P('data'))
replicated_sharding = NamedSharding(mesh, P())

# Wrap training loop in mesh context
with jax.set_mesh(mesh):
    for batch in data_loader:
        # Shard batch data
        dna_seq = jax.device_put(batch['dna_sequence'], data_sharding)
        organism = jax.device_put(batch['organism_index'], data_sharding)
        targets = jax.device_put(batch['targets'], data_sharding)
        
        loss, predictions, grad_norm = model.train_step(
            dna_seq, organism, targets, loss_fn
        )
''')

print("\n⚠️ KEY CHANGES:")
print("  1. Use mesh_utils.create_device_mesh()")
print("  2. Change axis name to 'data'")
print("  3. Simplify sharding to P('data')")
print("  4. Wrap training loop in 'with jax.set_mesh(mesh):'")
print("  5. Put batch on device INSIDE mesh context")

### Update `models/model.py` (HeadReplaceAlphaGenomeModel):

In [ ]:
print("IN models/model.py, update the __init__ sharding setup:")
print("="*60)
print('''
# Simplified sharding specs (around line 510)
data_sharding = NamedSharding(mesh, P('data'))  # Simplified!
replicated = NamedSharding(mesh, P())

self._train_step_jit = jax.jit(
    self._train_step,
    static_argnames=['loss_fn'],
    in_shardings=(
        data_sharding,                       # dna_sequence (auto-broadcasts to [B,S,4])
        data_sharding,                       # organism_index (auto-broadcasts to [B])
        data_sharding,                       # targets (auto-broadcasts to shape)
        replicated,                          # params
        replicated,                          # state
        replicated,                          # opt_state
    ),
    out_shardings=(
        replicated,                          # loss_value
        replicated,                          # predictions
        replicated,                          # new_params
        replicated,                          # new_opt_state
        replicated,                          # grad_norm
    )
)
''')

print("\n⚠️ KEY CHANGES:")
print("  1. Simplify all input shardings to P('data')")
print("  2. JAX will auto-broadcast P('data') to match input shapes")
print("  3. Specify replicated for predictions (not None)")

## 6. Action Plan

### Priority 1: Fix CRITICAL issues (Est. 30 min)

1. **Add `jax.set_mesh()` context manager** to training loop in `scripts/pytorch_ft.py`
2. **Simplify sharding specs** from `P('batch', None, None)` to `P('data')` in `models/model.py`
3. **Use `mesh_utils.create_device_mesh()`** for device mesh creation

### Priority 2: Test incrementally (Est. 1 hour)

1. Test with `debug_with_eval_shape.py` - should still pass
2. Run short training test (10-20 batches) to verify no NCCL errors
3. Monitor GPU memory and check for NCCL error messages

### Priority 3: Full validation (Est. 2-3 hours)

1. Run training for 1 epoch to ensure stability
2. Compare loss curves with previous attempts
3. Verify data isn't corrupted (check prediction ranges)

### Expected Outcome:

✅ NCCL errors should be resolved  
✅ Multi-GPU training should work smoothly  
✅ Training time should be ~25 hours for full run (8 GPUs)

## Summary

**Main Findings:**

1. ⚠️⚠️ **CRITICAL:** Missing `jax.set_mesh()` context manager
2. ⚠️⚠️ **CRITICAL:** Over-specified sharding with `P('batch', None, None)`
3. ⚠️ **MEDIUM:** Not using `mesh_utils.create_device_mesh()`
4. ✓ **LOW:** Axis name difference ('batch' vs 'data') - cosmetic only

**Root Cause:**
The missing `jax.set_mesh()` context manager means JAX doesn't know the active mesh when performing operations. This likely causes JAX to insert implicit resharding operations that trigger NCCL AllReduce calls with incorrect buffer configurations, leading to the "corrupted comm object" errors.

**Next Steps:**

1. Update `scripts/pytorch_ft.py` with all three fixes
2. Update `models/model.py` with simplified sharding
3. Run `debug_with_eval_shape.py` to verify shapes still work
4. Test short training run (20 batches)
5. If successful, proceed with full training

**Confidence Level:** HIGH - The missing `jax.set_mesh()` is a documented requirement for multi-GPU JAX training and matches DeepMind's working implementation exactly.

## 1. Official DeepMind Mesh Setup

From the official AlphaGenome finetuning notebook (`scripts/ag-colab_ft.py`):

# Mesh Setup Comparison: Your Implementation vs. Official DeepMind Notebook

This notebook compares your current JAX mesh setup for multi-GPU training with the official DeepMind AlphaGenome finetuning notebook.

**Goal:** Identify differences causing NCCL errors in your implementation.

**Status:** `jax.eval_shape` validation **PASSED ✓** - shapes are compatible, issue is in mesh/sharding configuration.